In [1]:
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import TextLoader

import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [3]:
llm = OpenAI(
    temperature=0,
    api_key=OPENAI_API_KEY
)

In [4]:
print(OPENAI_API_KEY[:10])

sk-proj-Nd


In [6]:
doc_path = "./datasets/CV_FML_ENG.md"

loader = TextLoader(
    doc_path,
    encoding="utf-8"
)

documents = loader.load()

In [7]:
type(documents)

list

In [8]:
len(documents)

1

In [9]:
documents[0]

Document(metadata={'source': './datasets/CV_FML_ENG.md'}, page_content='## FELIPE MARTIGNON LIZARRALDE\n\nCUSTOMER SUCCESS MANAGER | B2B SAAS | CUSTOMER ADOPTION\n\n+34 645262650\n\nCustomer Success professional with international experience helping B2B SaaS and fintech customers maximize business value through onboarding, product adoption, stakeholder management, and long-term customer success. Passionate about building trusted client relationships, using data and KPIs to measure customer outcomes, and acting as a consultative partner to help customers optimize workflows, improve adoption, and achieve measurable business results. Currently completing an AI Engineering Bootcamp, bringing additional knowledge of AI technologies, LLMs, AI workflows, APIs, and data-driven automation.\n\nmartignon@gmail.com\n\nhttps://www.linkedin.com/in/fmartignon/\n\n## PROFILE\n\n## WORK EXPERIENCE\n\n## CUSTOMER SUPPORT ANALYST\n\nKraken Crypto Exchange, Remote, Feb 2026 – Jun 2026\n\nManaged a high vo

In [10]:
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0
)

texts = text_splitter.split_documents(documents)

In [11]:
len(texts)

5

In [12]:
texts[0]

Document(metadata={'source': './datasets/CV_FML_ENG.md'}, page_content='## FELIPE MARTIGNON LIZARRALDE\n\nCUSTOMER SUCCESS MANAGER | B2B SAAS | CUSTOMER ADOPTION\n\n+34 645262650\n\nCustomer Success professional with international experience helping B2B SaaS and fintech customers maximize business value through onboarding, product adoption, stakeholder management, and long-term customer success. Passionate about building trusted client relationships, using data and KPIs to measure customer outcomes, and acting as a consultative partner to help customers optimize workflows, improve adoption, and achieve measurable business results. Currently completing an AI Engineering Bootcamp, bringing additional knowledge of AI technologies, LLMs, AI workflows, APIs, and data-driven automation.\n\nmartignon@gmail.com\n\nhttps://www.linkedin.com/in/fmartignon/\n\n## PROFILE\n\n## WORK EXPERIENCE\n\n## CUSTOMER SUPPORT ANALYST\n\nKraken Crypto Exchange, Remote, Feb 2026 – Jun 2026')

In [13]:
texts[1]

Document(metadata={'source': './datasets/CV_FML_ENG.md'}, page_content='Managed a high volume of customer relationships involving account security, authentication, product usage, and payment-related issues, helping customers maximize product adoption while building trust and ensuring long-term customer satisfaction. Collaborated with Product, Operations, and Compliance teams to resolve complex customer challenges, improve service delivery, and translate technical concepts into practical guidance that increased customer confidence and product adoption.\n\n## CLIENT OPERATIONS & SUPPORT SPECIALIST\n\nAPEX - FundRock LIS, Valencia, Jul 2025 – Feb 2026')

In [15]:
embeddings = OpenAIEmbeddings(
    api_key=OPENAI_API_KEY
)

In [16]:
docsearch = Chroma.from_documents(
    texts,
    embeddings,
    collection_name="felipe-cv"
)

In [17]:
embeddings = OpenAIEmbeddings(
    api_key=OPENAI_API_KEY
)

In [18]:
docsearch = Chroma.from_documents(
    texts,
    embeddings,
    collection_name="felipe-cv"
)

In [19]:
retriever = docsearch.as_retriever()

In [20]:
type(retriever)

langchain_core.vectorstores.base.VectorStoreRetriever

In [21]:
cv_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever
)

In [23]:
cv_qa.run("What languages does Felipe speak?")

' Felipe speaks Spanish as a native language, has full professional proficiency in English (C1/C2), and is intermediate in French (B1/B2).'

In [24]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://beta.ruff.rs/docs/faq/")
docs = loader.load()

ruff_texts = text_splitter.split_documents(docs)

ruff_db = Chroma.from_documents(
    ruff_texts,
    embeddings,
    collection_name="ruff"
)

ruff_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ruff_db.as_retriever()
)

USER_AGENT environment variable not set, consider setting it to identify your requests.
Created a chunk of size 2122, which is longer than the specified 1000
Created a chunk of size 3187, which is longer than the specified 1000
Created a chunk of size 1017, which is longer than the specified 1000
Created a chunk of size 2321, which is longer than the specified 1000


In [25]:
ruff_qa.run("Why use Ruff over Flake8?")

' Ruff offers a larger rule set and better compatibility with Black, making it a more comprehensive and efficient linter. It also has the added benefit of being able to automatically fix its own lint violations.'

In [26]:
from langchain.agents import Tool

tools = [
    Tool(
        name="Felipe CV Assistant",
        func=cv_qa.run,
        description=(
            "Useful for answering questions about Felipe Martignon's "
            "professional experience, education, technical skills, "
            "AI projects, certifications, languages and work history. "
            "Input should be a complete question."
        ),
    ),
    Tool(
        name="Ruff Documentation",
        func=ruff_qa.run,
        description=(
            "Useful for answering questions about Ruff, "
            "the Python linter, including installation, "
            "features and best practices."
        ),
    ),
]

In [27]:
from langchain.agents import initialize_agent, AgentType

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

/var/folders/db/87d025j5081fm4bs5t49d62r0000gn/T/ipykernel_23144/3751267280.py:3: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


In [28]:
agent.invoke(
    "What programming languages does Felipe know?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 We can use Felipe CV Assistant to answer this question.
Action: Felipe CV Assistant
Action Input: "What programming languages does Felipe know?"
Observation:  It is not specified in the given context what programming languages Felipe knows.
Thought: We can also use Ruff Documentation to answer this question.
Action: Ruff Documentation
Action Input: "What programming languages does Felipe know?"
Observation:  I don't know, as the context provided does not mention anything about Felipe or his knowledge of programming languages.
Thought: I now know the final answer.
Final Answer: I cannot answer this question as there is not enough information provided about Felipe's programming language knowledge.

> Finished chain.


{'input': 'What programming languages does Felipe know?',
 'output': "I cannot answer this question as there is not enough information provided about Felipe's programming language knowledge."}

In [29]:
agent.invoke(
    "Based on Felipe's experience, would Ruff be useful in his projects? Explain why."
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should check Felipe's technical skills and AI projects to determine if Ruff would be useful.
Action: Felipe CV Assistant
Action Input: Felipe Martignon
Observation:  Felipe Martignon is a Customer Success Manager with international experience in B2B SaaS and fintech industries. He is currently completing an AI Engineering Bootcamp and has a background in business administration, economics, and sports management. He has also worked as a Customer Support Analyst, Assistant Manager, and Customer Service Specialist in various companies.
Thought: Felipe has experience in both technical and non-technical roles, so Ruff could potentially be useful in his projects.
Action: Felipe CV Assistant
Action Input: Felipe Martignon
Observation:  Felipe Martignon is a Customer Success Manager with international experience in B2B SaaS and fintech industries. He is currently completing an AI Engineering Bootcamp and has a background in business administration, economics, and sports management. He has a

{'input': "Based on Felipe's experience, would Ruff be useful in his projects? Explain why.",
 'output': "Yes, Ruff could be useful in Felipe's projects, especially if he is using Python and wants to ensure code quality and consistency."}

In [30]:
agent.invoke(
    "What languages does Felipe speak?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 We can use Felipe CV Assistant to answer this question.
Action: Felipe CV Assistant
Action Input: "What languages does Felipe speak?"
Observation:  Felipe speaks Spanish as a native language, has full professional proficiency in English (C1/C2), and is intermediate in French (B1/B2).
Thought: We can also check for any additional languages mentioned in his education or work history.
Action: Felipe CV Assistant
Action Input: "What languages are mentioned in Felipe's education and work history?"
Observation:  Spanish, English, and French.
Thought: I now know the final answer.
Final Answer: Felipe speaks Spanish, English, and French.

> Finished chain.


{'input': 'What languages does Felipe speak?',
 'output': 'Felipe speaks Spanish, English, and French.'}